In [2]:
import pyodbc
import pandas as pd
import time
import os
from tqdm.auto import tqdm
import threading
import itertools
import sys

# 🔐 Подключение
uid = "ii_user"
pwd = "Vjoi970N"
driver = "{ODBC Driver 18 for SQL Server}"
server = "192.168.200.196,59197"
database = "dwh_price"

SCHEMA_NAME = "dbo"
VIEW_NAME = "AI_stock_farm_market_table"

# 📁 Папка для файлов
OUTPUT_DIR = r"C:\Проекты\Project_etl_power_bi\data\Problems\ai_stock_8years_chunks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 📆 Диапазон дат, который хочешь выгрузить
START_DATE = "2025-02-01"
END_DATE   = "2026-02-09"  # включительно



# ============================================================
# 🎛 Анимация "запрос выполняется..."
# ============================================================
class Spinner:
    def __init__(self, message="⏳ Выполняется SQL-запрос... "):
        self.spinner = itertools.cycle(["|", "/", "-", "\\"])
        self.running = False
        self.thread = None
        self.message = message

    def start(self):
        self.running = True
        self.thread = threading.Thread(target=self.animate)
        self.thread.start()

    def animate(self):
        while self.running:
            sys.stdout.write(f"\r{self.message}{next(self.spinner)}")
            sys.stdout.flush()
            time.sleep(0.2)

    def stop(self):
        self.running = False
        if self.thread:
            self.thread.join()
        sys.stdout.write("\r")  # очистить строку


# ============================================================
# 🚀 Основной код
# ============================================================
def main():
    # -----------------------------
    # Генерируем месячные периоды
    # -----------------------------
    start_dt = pd.to_datetime(START_DATE)
    # верхняя граница эксклюзивная: берем END_DATE + 1 день
    end_dt_excl = pd.to_datetime(END_DATE) + pd.Timedelta(days=1)

    # Начинаем с первого числа месяца START_DATE
    first_month_start = start_dt.replace(day=1)

    # Месячные границы от первого числа стартового месяца до end_dt_excl
    month_starts = pd.date_range(start=first_month_start, end=end_dt_excl, freq="MS")

    periods = []
    for i, m_start in enumerate(month_starts):
        # Старт периода: для первого берём именно START_DATE (может быть не первое число)
        if i == 0:
            s = start_dt
        else:
            s = m_start

        # Конец периода: для всех, кроме последнего — следующий месяц;
        # для последнего — end_dt_excl (чтобы захватить не полный месяц)
        if i < len(month_starts) - 1:
            e = month_starts[i + 1]
        else:
            e = end_dt_excl

        periods.append((s.strftime("%Y-%m-%d"), e.strftime("%Y-%m-%d")))

    print("📆 Периоды для выгрузки:")
    for i, (s, e) in enumerate(periods, start=1):
        print(f"{i:02d}: {s} → {e}")
    print()

    conn = None
    try:
        conn = pyodbc.connect(
            f"DRIVER={driver};SERVER={server};DATABASE={database};"
            f"UID={uid};PWD={pwd};TrustServerCertificate=yes;",
            autocommit=True,
        )
        print("✅ Соединение с БД установлено\n")

        

        total_rows = 0
        total_start = time.time()

        # tqdm — прогресс по периодам (месяцы + последний неполный)
        for idx, (start_date, end_date) in enumerate(
            tqdm(periods, desc="📊 Периоды", unit="period"), start=1
        ):
            print(f"\n📆 Обрабатываю период {idx:02d}: {start_date} → {end_date}")

            sql = f"""
            SELECT
                *
            FROM [{SCHEMA_NAME}].[{VIEW_NAME}] WITH (NOLOCK)
            WHERE [Дата] >= '{start_date}'
              AND [Дата] < '{end_date}'
              OPTION (MAXDOP 15)
            """

            # ⏳ Запускаем анимацию "запрос выполняется"
            spinner = Spinner()
            spinner.start()
            t0 = time.time()

            df_part = pd.read_sql(sql, conn)

            spinner.stop()
            dt = time.time() - t0

            rows = len(df_part)
            total_rows += rows

            print(f"   📥 Получено строк: {rows:,}")
            print(f"   ⏱ SQL-время: {dt:.1f} сек")

            # если пусто — не сохраняем
            if rows == 0:
                print("   ⚠️ Нет данных, пропускаю.\n")
                continue

            filename = f"ai_stock_{start_date}_to_{end_date}.csv".replace("-", "")
            path = os.path.join(OUTPUT_DIR, filename)
            df_part.to_csv(path, index=False)

            print(f"   💾 Сохранено в файл: {path}\n")

        total_dt = time.time() - total_start
        print("✅ ВЫГРУЗКА ЗАВЕРШЕНА!")
        print(f"📊 Всего выгружено строк: {total_rows:,}")
        print(f"⏱ Общее время: {total_dt/60:.1f} мин")

    except Exception as e:
        print("\n❌ Ошибка при выгрузке:")
        print(e)

    finally:
        if conn:
            conn.close()
            print("\n🔚 Соединение с БД закрыто")


if __name__ == "__main__":
    main()


📆 Периоды для выгрузки:
01: 2025-02-01 → 2025-03-01
02: 2025-03-01 → 2025-04-01
03: 2025-04-01 → 2025-05-01
04: 2025-05-01 → 2025-06-01
05: 2025-06-01 → 2025-07-01
06: 2025-07-01 → 2025-08-01
07: 2025-08-01 → 2025-09-01
08: 2025-09-01 → 2025-10-01
09: 2025-10-01 → 2025-11-01
10: 2025-11-01 → 2025-12-01
11: 2025-12-01 → 2026-01-01
12: 2026-01-01 → 2026-02-01
13: 2026-02-01 → 2026-02-10

✅ Соединение с БД установлено



📊 Периоды:   0%|          | 0/13 [00:00<?, ?period/s]


📆 Обрабатываю период 01: 2025-02-01 → 2025-03-01
⏳ Выполняется SQL-запрос... -

C:\Users\SIMukovoz\AppData\Local\Temp\ipykernel_32144\1859700376.py:131: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_part = pd.read_sql(sql, conn)


   📥 Получено строк: 427,298
   ⏱ SQL-время: 5.3 сек
   💾 Сохранено в файл: C:\Проекты\Project_etl_power_bi\data\Problems\ai_stock_8years_chunks\ai_stock_20250201_to_20250301.csv


📆 Обрабатываю период 02: 2025-03-01 → 2025-04-01
   📥 Получено строк: 453,181
   ⏱ SQL-время: 5.2 сек
   💾 Сохранено в файл: C:\Проекты\Project_etl_power_bi\data\Problems\ai_stock_8years_chunks\ai_stock_20250301_to_20250401.csv


📆 Обрабатываю период 03: 2025-04-01 → 2025-05-01
   📥 Получено строк: 464,128
   ⏱ SQL-время: 5.5 сек
   💾 Сохранено в файл: C:\Проекты\Project_etl_power_bi\data\Problems\ai_stock_8years_chunks\ai_stock_20250401_to_20250501.csv


📆 Обрабатываю период 04: 2025-05-01 → 2025-06-01
   📥 Получено строк: 424,417
   ⏱ SQL-время: 4.9 сек
   💾 Сохранено в файл: C:\Проекты\Project_etl_power_bi\data\Problems\ai_stock_8years_chunks\ai_stock_20250501_to_20250601.csv


📆 Обрабатываю период 05: 2025-06-01 → 2025-07-01
   📥 Получено строк: 433,231
   ⏱ SQL-время: 5.1 сек
   💾 Сохранено в файл: C:\П

In [3]:
import glob

In [4]:
# 📁 Папка, где лежат parquet-чанки
CHUNKS_DIR = r"C:\Проекты\Project_etl_power_bi\data\Problems\ai_stock_8years_chunks"

def load_all_chunks_to_one_df(chunks_dir: str) -> pd.DataFrame:
    pattern = os.path.join(chunks_dir, "*.csv")
    files = sorted(glob.glob(pattern))

    if not files:
        raise FileNotFoundError(f"Не найдено ни одного файла по маске: {pattern}")

    print(f"Найдено csv-файлов: {len(files)}")

    dfs = []
    for file in files:
        df_chunk = pd.read_csv(file)
        dfs.append(df_chunk)

    df_full = pd.concat(dfs, ignore_index=True)

    print(f"Итоговый размер: {df_full.shape[0]:,} строк × {df_full.shape[1]} колонок")
    mem_mb = df_full.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"Память: ~{mem_mb:.2f} MB")

    return df_full


if __name__ == "__main__":
    df = load_all_chunks_to_one_df(CHUNKS_DIR)


Найдено csv-файлов: 13
Итоговый размер: 5,563,263 строк × 19 колонок
Память: ~3805.91 MB


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5563263 entries, 0 to 5563262
Data columns (total 19 columns):
 #   Column                                Dtype  
---  ------                                -----  
 0   Дата                                  object 
 1   CAG_ID                                object 
 2   Код КАГ                               int64  
 3   Имя КАГ                               object 
 4   Активный КАГ                          object 
 5   Код товара                            object 
 6   Наименование у нас                    object 
 7   Остаток общий                         int64  
 8   Количество Ирбит                      int64  
 9   Остаток в пути                        int64  
 10  ГК (остатки гранд капитала)           int64  
 11  Пульс (остатки пульса)                int64  
 12  Катрен (остатки катрена)              int64  
 13  Протек (остатки протека)              int64  
 14  Фармкомплект (остатки фармкомплекта)  int64  
 15  Цена пульса    

In [7]:
df[df['Дата'] == '2026-02-09']

,Дата,CAG_ID,Код КАГ,Имя КАГ,Активный КАГ,Код товара,Наименование у нас,Остаток общий,Количество Ирбит,Остаток в пути,ГК (остатки гранд капитала),Пульс (остатки пульса),Катрен (остатки катрена),Протек (остатки протека),Фармкомплект (остатки фармкомплекта),Цена пульса,Цена катрена,Цена протека,Цена фармкомплекта
5545055,2026-02-09,8800000000011726,11540,КОНТЕКС презервативы Classic (классические) №12,Да,016501,КОНТЕКС презервативы Classic (классические) №12,0,0,0,0,20047,0,19618,303,533.74,0.00,539.44,389.56
5545056,2026-02-09,8800000000025494,21594,"Санорин спрей наз.(ментол/эвкалипт) 0,1% фл. 1...",Да,028025,Санорин спрей наз.(ментол/эвкалипт) 0.1% фл. 1...,921,0,0,921,0,0,0,0,0.00,0.00,0.00,0.00
5545057,2026-02-09,8800000000035053,29530,Sense of Life Омега-3 Здоровье всего организма...,Нет,050763,Sense of Life Омега-3 Здоровье всего организма...,0,0,0,0,0,0,40,20,0.00,0.00,623.93,607.00
5545058,2026-02-09,8800000000010256,8488,Три-Мерси таб. п/п/о №21,Нет,030676,Три-Мерси таб. п/п/о №21,0,0,0,0,0,217,140,0,0.00,1066.94,1079.31,0.00
5545059,2026-02-09,8800000000029632,23780,Железа препарат Феррумтабс Сердце Континента т...,Да,035341,Железа пр-т Феррумтабс Сердце Континента т.п/о№30,4245,0,0,4245,0,0,0,0,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563258,2026-02-09,8800000000021559,19741,Глимепирид Канон таб. 1мг №30,Нет,022642,Глимепирид Канон таб. 1мг №30,0,0,0,0,524,0,0,0,160.11,0.00,0.00,0.00
5563259,2026-02-09,880000000000E2B8,394,Альбумин р-р д/инф. 10% 100мл фл. №1,Да,2-000575,Альбумин р-р д/инф. 10% 100мл фл. №1,0,0,0,0,0,564,0,0,0.00,3658.90,0.00,0.00
5563260,2026-02-09,8800000000016FC4,18309,Гематоген Русский с кокосом в глазури 40г,Да,019309,Гематоген Русский с кокосом в глазури 40г,2091,0,0,2091,0,338,2,0,0.00,22.60,19.35,0.00
5563261,2026-02-09,88000000000198B4,18837,Реглисам таб. 50мг №50,Да,020405,Реглисам таб. 50мг №50,982,0,0,982,0,1019,1965,0,0.00,901.09,917.27,0.00


In [9]:
import pandas as pd

In [23]:
df = pd.read_parquet(r'C:\Проекты\Project_etl_power_bi\data\preproc_parquet\big_data_clean.parquet')

In [24]:
df['Дата'].max()

Timestamp('2026-02-09 00:00:00')

In [25]:
df[df['Дата'] == df['Дата'].max()]

,Код КАГ,Дата,ГК (остатки гранд капитала),Пульс (остатки пульса),Катрен (остатки катрена),Протек (остатки протека),Фармкомплект (остатки фармкомплекта),Цена пульса,Цена катрена,Цена протека,Цена фармкомплекта,weekday,Имя КАГ,Код товара,Наименование у нас
257,10000,2026-02-09,1436,1445,367,654,1664,745.80,753.21,744.93,718.89,0,Персен Ночь капс. №40,009680; 013536; 045883,Персен ночной капс. №40; Персен Ночь капс. №40
517,10001,2026-02-09,12757,6856,1006,44897,0,256.03,268.69,262.63,0.00,0,Перинева таб. 4мг №30,009442,Перинева таб. 4мг №30
777,10002,2026-02-09,2655,63537,366,21923,5637,276.40,288.46,288.50,269.51,0,Перинева таб. 8мг №30,009464,Перинева таб. 8мг №30
1037,10003,2026-02-09,4641,2239,3376,25036,2100,636.31,651.97,619.33,599.88,0,Перинева таб. 4мг №90,009435; 050577,Перинева таб. 4мг №90
1297,10004,2026-02-09,3218,3182,604,14770,3488,710.27,726.48,742.86,701.52,0,Перинева таб. 8мг №90,009573,Перинева таб. 8мг №90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2090521,9989,2026-02-09,168,183,624,1211,1289,116.66,119.95,112.82,108.45,0,Нипертен таб. п/п/о 2.5мг №30,009744,Нипертен таб. п/п/о 2.5мг №30
2090781,9990,2026-02-09,1965,1708,489,862,40,311.08,321.11,312.74,292.44,0,Нипертен таб. п/п/о 5мг №100,009562,Нипертен таб. п/п/о 5мг №100
2091041,9991,2026-02-09,2655,3689,661,9087,0,281.18,284.12,282.50,0.00,0,Нурофен Экспресс гель д/нар. прим. 5% туба 100г,014129; 009484,Нурофен Экспресс гель д/нар. прим. 5% туба 100г
2091301,9992,2026-02-09,5051,15823,1218,24287,0,140.69,142.01,141.33,0.00,0,Нурофен Экспресс гель д/нар. прим. 5% туба 50г,009466,Нурофен Экспресс гель д/нар. прим. 5% туба 50г
